In [61]:
import os
import json
import random
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from typing import Dict, Tuple, Optional
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score


In [62]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [63]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")
meta_artifacts_dir = os.path.join(base_dir, "meta_artifacts")
checkpoints_dir = os.path.join(base_dir, "checkpoints")

os.makedirs(checkpoints_dir, exist_ok=True)

print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("fusion_artifacts_dir:", fusion_artifacts_dir)
print("meta_artifacts_dir:", meta_artifacts_dir)
print("checkpoints_dir:", checkpoints_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts
meta_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/meta_artifacts
checkpoints_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints


In [64]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Reproducibility settings applied (seed =", SEED, ")")

Reproducibility settings applied (seed = 42 )


In [65]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB


In [66]:
# Load configuration files
fusion_config_path = os.path.join(fusion_artifacts_dir, "fusion_config.json")
meta_config_path = os.path.join(meta_artifacts_dir, "meta_config.json")

assert os.path.exists(fusion_config_path), f"Missing fusion_config.json at {fusion_config_path}"
assert os.path.exists(meta_config_path), f"Missing meta_config.json at {meta_config_path}"

with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

with open(meta_config_path, "r") as f:
    meta_config = json.load(f)

In [67]:
print("--- Loaded fusion_config.json ---")
for k in ["num_users", "num_items", "num_scenes", "d_model", "num_experts", "hidden_dim"]:
    print(f"{k}: {fusion_config[k]}")

--- Loaded fusion_config.json ---
num_users: 67615
num_items: 59142
num_scenes: 5
d_model: 256
num_experts: 4
hidden_dim: 256


In [68]:
print("--- Loaded meta_config.json ---")
for k in ["meta_dim", "FEWSHOT_N", "blend_strategy", "cold_rule"]:
    print(f"{k}: {meta_config[k]}")

--- Loaded meta_config.json ---
meta_dim: 256
FEWSHOT_N: 10
blend_strategy: deterministic_alpha
cold_rule: item_only_fewshot


### Load Core Training Artifacts

Here we will loads the precomputed, row-aligned artifacts required for end-to-end CTR training. Interaction-level features (IDs, text embeddings, and time features) must have identical length to guarantee that each row index refers to the same user–item–scene interaction across all modalities. Item-level image embeddings are loaded separately and indexed via item_id_idx.

In [69]:
# paths to artifacts
df_id_path = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.pkl")
text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")

image_embeddings_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

splits_dir = os.path.join(preprocessed_dir, "splits")
train_idx_path = os.path.join(splits_dir, "train_idx.npy")
val_idx_path   = os.path.join(splits_dir, "val_idx.npy")
test_idx_path  = os.path.join(splits_dir, "test_idx.npy")

In [70]:
# validate file existance
required_paths = {
    "df_id_encoded.pkl": df_id_path,
    "text_embeddings.npy": text_embeddings_path,
    "time_features_final.pt": time_features_path,
    "image_embeddings_vit_base_aligned.pt": image_embeddings_path,
    "image_missing_mask.pt": image_missing_mask_path,
    "train_idx.npy": train_idx_path,
    "val_idx.npy": val_idx_path,
    "test_idx.npy": test_idx_path
}

for name, path in required_paths.items():
    assert os.path.exists(path), f"Missing {name} at: {path}"

print("All required artifacts found.")

All required artifacts found.


In [71]:
# Load interaction-level IDs (row-aligned)
df_id = pd.read_pickle(df_id_path)
print("Loaded df_id_encoded.pkl:", df_id.shape)
print("Columns:", df_id.columns.tolist())
print(df_id.head())

Loaded df_id_encoded.pkl: (3730329, 3)
Columns: ['user_id_idx', 'item_id_idx', 'scene_idx']
   user_id_idx  item_id_idx  scene_idx
0        41688        13590          4
1        39834        21351          2
2        22354        33398          2
3        17547        11190          1
4        55873        33398          2


In [72]:
# Load text embeddings as memmap (interaction-level)
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
print("Loaded text_embeddings.npy (memmap):", text_embeddings.shape, "| dtype:", text_embeddings.dtype)

Loaded text_embeddings.npy (memmap): (3730329, 1536) | dtype: float32


In [73]:
# Load time features (interaction-level)
time_features = torch.load(time_features_path, map_location="cpu")
print("Loaded time_features_final.pt:", tuple(time_features.shape), "| dtype:", time_features.dtype)

Loaded time_features_final.pt: (3730329, 5) | dtype: torch.float32


In [74]:
# Load image embeddingd + missing mask (item-level)
image_embeddings = torch.load(image_embeddings_path, map_location="cpu")
image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")

print("Loaded image_embeddings:", tuple(image_embeddings.shape), "| dtype:", image_embeddings.dtype)
print("Loaded image_missing_mask:", tuple(image_missing_mask.shape), "| missing count:", int(image_missing_mask.sum().item()))

Loaded image_embeddings: (59142, 768) | dtype: torch.float32
Loaded image_missing_mask: (59142,) | missing count: 2


In [75]:
# Load deterministic splits
train_idx = np.load(train_idx_path)
val_idx   = np.load(val_idx_path)
test_idx  = np.load(test_idx_path)

print("Loaded splits:")
print("train:", len(train_idx), "val:", len(val_idx), "test:", len(test_idx))

Loaded splits:
train: 2984263 val: 373033 test: 373033


In [76]:
N = len(df_id)

assert text_embeddings.shape[0] == N, f"text_embeddings rows ({text_embeddings.shape[0]}) != df_id rows ({N})"
assert time_features.shape[0] == N, f"time_features rows ({time_features.shape[0]}) != df_id rows ({N})"

EXPECTED_N = 3_730_329
assert N == EXPECTED_N, f"Unexpected df_id row count: {N} (expected {EXPECTED_N})"

# Split indices bounds check
assert train_idx.max() < N and val_idx.max() < N and test_idx.max() < N, "Split indices exceed dataset length"

print("All artifacts loaded and row-alignment verified.")
print(f"Interaction rows N = {N:,} (expected {EXPECTED_N:,})")

All artifacts loaded and row-alignment verified.
Interaction rows N = 3,730,329 (expected 3,730,329)


### Dataset and DataLoader

*   This section defines a PyTorch CTRDataset that returns all model inputs in a sample format.
*   Each sample is indexed by an interaction row_idx and returns the aligned ID indices (user/item/scene), interaction-level text and time features, item-level image embeddings (with missing images handled safely), and the click label.
*  A deterministic cold-start mask is also created using train-only frequency counts (few-shot rule), to allow cold/warm evaluation during training.

In [77]:
# Load labels (click) from a row-aligned source
label = None
label_source = None

# df_processed.pkl (contains label column)
df_processed_path = os.path.join(preprocessed_dir, "df_processed.pkl")
if os.path.exists(df_processed_path):
    df_processed = pd.read_pickle(df_processed_path)
    for cand in ["label", "is_click", "click", "y"]:
        if cand in df_processed.columns:
            label = df_processed[cand].to_numpy(dtype=np.float32)
            label_source = f"df_processed.pkl[{cand}]"
            break

In [78]:
assert label is not None, (
    "Could not find a label column. Expected one of ['label','is_click','click','y'] in "
    "df_processed.pkl or split_features/df_deep.csv. Please confirm where the click label lives."
)

assert len(label) == len(df_id), f"Label length {len(label)} != df_id rows {len(df_id)}"
print(f"Loaded label from: {label_source} | shape={label.shape} | positives={label.mean():.6f}")

Loaded label from: df_processed.pkl[label] | shape=(3730329,) | positives=0.336333


In [79]:
# Train-only frequency counts (for few-shot cold mask)
num_items = fusion_config["num_items"]
FEWSHOT_N = int(meta_config["FEWSHOT_N"])  # reuse what was saved in Notebook 07

train_items = df_id.iloc[train_idx]["item_id_idx"].to_numpy(dtype=np.int64)
item_train_count = np.bincount(train_items, minlength=num_items)

# Few-shot definition: seen in train but < FEWSHOT_N exposures
item_is_fewshot = (item_train_count > 0) & (item_train_count < FEWSHOT_N)
item_is_warm = (item_train_count >= FEWSHOT_N)

print("--- Train-only item frequency summary ---")
print("FEWSHOT_N:", FEWSHOT_N)
print("Few-shot items:", int(item_is_fewshot.sum()), "/", num_items)
print("Warm items:", int(item_is_warm.sum()), "/", num_items)

--- Train-only item frequency summary ---
FEWSHOT_N: 10
Few-shot items: 39751 / 59142
Warm items: 14898 / 59142


In [80]:
# convert mask to torch for fast indexing in Dataset
item_is_fewshot_torch = torch.tensor(item_is_fewshot, dtype=torch.bool)

In [81]:
class CTRDataset(Dataset):
    """
    Row-aligned CTR dataset.

    Returns:
      user_id_idx: LongTensor ()
      item_id_idx: LongTensor ()
      scene_idx:   LongTensor ()
      text_vec:    FloatTensor (1536)
      time_vec:    FloatTensor (5)
      image_vec:   FloatTensor (768) (zero if missing)
      label:       FloatTensor ()
      cold_mask:   BoolTensor ()
    """
    def __init__(
        self,
        row_indices: np.ndarray,
        df_id: pd.DataFrame,
        text_embeddings_memmap: np.ndarray,
        time_features: torch.Tensor,
        image_embeddings: torch.Tensor,
        image_missing_mask: torch.Tensor,
        labels: np.ndarray,
        item_is_fewshot: torch.Tensor
    ):
        self.row_indices = row_indices.astype(np.int64)
        self.df_id = df_id
        self.text_mem = text_embeddings_memmap
        self.time = time_features                    # CPU tensor (N,5)
        self.image = image_embeddings                # CPU tensor (num_items,768)
        self.img_missing = image_missing_mask.bool() # CPU tensor (num_items,)
        self.labels = labels                         # numpy (N,)
        self.item_is_fewshot = item_is_fewshot       # torch bool (num_items,)

        # Cache dims
        self.text_dim = self.text_mem.shape[1]
        self.time_dim = self.time.shape[1]
        self.image_dim = self.image.shape[1]

    def __len__(self):
        return len(self.row_indices)

    def __getitem__(self, i: int):
        row_idx = int(self.row_indices[i])

        row = self.df_id.iloc[row_idx]
        user_idx = int(row["user_id_idx"])
        item_idx = int(row["item_id_idx"])
        scene_idx = int(row["scene_idx"])

        # Interaction-level features
        text_vec = torch.tensor(self.text_mem[row_idx], dtype=torch.float32)  # (1536,)
        time_vec = self.time[row_idx].float()                                 # (5,)

        # Item-level image embedding (zero if missing)
        if bool(self.img_missing[item_idx].item()):
            image_vec = torch.zeros(self.image_dim, dtype=torch.float32)
        else:
            image_vec = self.image[item_idx].float()                  # (768,)

        # Label
        y = float(self.labels[row_idx])
        label = torch.tensor(y, dtype=torch.float32)

        # Cold mask (item-only few-shot rule)
        cold_mask = self.item_is_fewshot[item_idx]  # bool scalar

        return {
            "row_idx": torch.tensor(row_idx, dtype=torch.long),
            "user_idx": torch.tensor(user_idx, dtype=torch.long),
            "item_idx": torch.tensor(item_idx, dtype=torch.long),
            "scene_idx": torch.tensor(scene_idx, dtype=torch.long),
            "text_vec": text_vec,
            "time_vec": time_vec,
            "image_vec": image_vec,
            "label": label,
            "cold_mask": cold_mask
        }


def make_loader(row_indices: np.ndarray, batch_size: int, shuffle: bool):
    ds = CTRDataset(
        row_indices=row_indices,
        df_id=df_id,
        text_embeddings_memmap=text_embeddings,
        time_features=time_features,
        image_embeddings=image_embeddings,
        image_missing_mask=image_missing_mask,
        labels=label,
        item_is_fewshot=item_is_fewshot_torch
    )

    pin = (device.type == "cuda")
    num_workers = 2

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2,
        drop_last=False
    )
    return ds, loader

In [82]:
train_ds, train_loader = make_loader(train_idx, batch_size=batch_size, shuffle=True)
batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items() if hasattr(v, "shape")})

{'row_idx': torch.Size([4096]), 'user_idx': torch.Size([4096]), 'item_idx': torch.Size([4096]), 'scene_idx': torch.Size([4096]), 'text_vec': torch.Size([4096, 1536]), 'time_vec': torch.Size([4096, 5]), 'image_vec': torch.Size([4096, 768]), 'label': torch.Size([4096]), 'cold_mask': torch.Size([4096])}


* Text embeddings are read via memmap, it avoids loading 3.7M × 1536 into RAM.
* Image embeddings are item-level and small enough to keep on CPU and index quickly. -> Can move image_vec to GPU per batch during training.
* num_workers=4 set as default on Colab; can increase to 8 only if it’s stable.

In [83]:
# Choose an initial batch size depending on device
batch_size = 4096 if device.type == "cuda" else 1024

train_ds, train_loader = make_loader(train_idx, batch_size=batch_size, shuffle=True)
val_ds, val_loader = make_loader(val_idx, batch_size=batch_size, shuffle=False)

print("Datasets created:")
print("Train size:", len(train_ds), "| Val size:", len(val_ds))
print("Batch size:", batch_size)

Datasets created:
Train size: 2984263 | Val size: 373033
Batch size: 4096


In [84]:
def assert_finite(t: torch.Tensor, name: str):
    assert torch.isfinite(t).all(), f"{name} has NaN/Inf"

# First batch sanity check
batch = next(iter(train_loader))

In [85]:
# Shapes
print("user_idx:", batch["user_idx"].shape, batch["user_idx"].dtype)
print("item_idx:", batch["item_idx"].shape, batch["item_idx"].dtype)
print("scene_idx:", batch["scene_idx"].shape, batch["scene_idx"].dtype)

print("text_vec:", batch["text_vec"].shape, batch["text_vec"].dtype)
print("time_vec:", batch["time_vec"].shape, batch["time_vec"].dtype)
print("image_vec:", batch["image_vec"].shape, batch["image_vec"].dtype)

print("label:", batch["label"].shape, batch["label"].dtype)
print("cold_mask:", batch["cold_mask"].shape, batch["cold_mask"].dtype)

user_idx: torch.Size([4096]) torch.int64
item_idx: torch.Size([4096]) torch.int64
scene_idx: torch.Size([4096]) torch.int64
text_vec: torch.Size([4096, 1536]) torch.float32
time_vec: torch.Size([4096, 5]) torch.float32
image_vec: torch.Size([4096, 768]) torch.float32
label: torch.Size([4096]) torch.float32
cold_mask: torch.Size([4096]) torch.bool


In [86]:
# NaN checks (float tensors)
assert_finite(batch["text_vec"], "text_vec")
assert_finite(batch["time_vec"], "time_vec")
assert_finite(batch["image_vec"], "image_vec")
assert_finite(batch["label"], "label")

In [87]:
# Quick rates
cold_rate = float(batch["cold_mask"].float().mean().cpu())
pos_rate = float(batch["label"].mean().cpu())
print(f"Batch cold rate: {cold_rate:.4%}")
print(f"Batch positive rate: {pos_rate:.4%}")

Batch cold rate: 3.6133%
Batch positive rate: 33.6670%


### End-to-End ColdFusionX Model Assembly

This section assembles the full trainable CTR prediction model for end-to-end optimisation. The model integrates:
1.   ID embeddings
2.   meta-embedding generator for cold-start generalisation
3.   α-based blending to unify cold/warm item representations
4.   modality attention fusion across ID/text/image/time features
5.   scenario-aware MMoE routing before producing a final CTR logit via a lightweight prediction head.

A forward-pass sanity test is then executed to validate tensor shapes, attention normalisation, and gate normalisation prior to training.

In [88]:
# from Notebook 06
class ModalityAttentionFusion(nn.Module):
    """
    Projects {id, text, image, time} into a shared d_model space and applies
    per-sample attention over modalities.

    Forward inputs:
      id_vec:    (B, id_dim)
      text_vec:  (B, text_dim=1536)
      image_vec: (B, image_dim=768)
      time_vec:  (B, time_dim=5 or Time2Vec dim later)

    Output:
      fused: (B, d_model)

    Diagnostics:
      get_last_attention() returns a dict of attention weights.
    """
    def __init__(
        self,
        id_dim: int,
        text_dim: int,
        image_dim: int,
        time_dim: int,
        d_model: int = 256,
        attn_hidden: int = 128,
        dropout: float = 0.1
    ):
        super().__init__()

        # Projections into shared d_model
        self.proj_id = nn.Sequential(
            nn.Linear(id_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.proj_text = nn.Sequential(
            nn.Linear(text_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.proj_image = nn.Sequential(
            nn.Linear(image_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.proj_time = nn.Sequential(
            nn.Linear(time_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Attention MLP produces 1 scalar score per modality per sample
        self.attn_mlp = nn.Sequential(
            nn.Linear(d_model, attn_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(attn_hidden, 1)
        )

        self._last_attention: Optional[Dict[str, torch.Tensor]] = None

    def forward(
        self,
        id_vec: torch.Tensor,
        text_vec: torch.Tensor,
        image_vec: torch.Tensor,
        time_vec: torch.Tensor
    ) -> torch.Tensor:

        # Project each modality to (B, d_model)
        z_id = self.proj_id(id_vec)
        z_text = self.proj_text(text_vec)
        z_image = self.proj_image(image_vec)
        z_time = self.proj_time(time_vec)

        # Stack: (B, M=4, d_model)
        Z = torch.stack([z_id, z_text, z_image, z_time], dim=1)

        # Compute attention logits per modality: (B, 4, 1) -> (B, 4)
        logits = self.attn_mlp(Z).squeeze(-1)

        # Softmax over modalities -> (B, 4)
        attn = F.softmax(logits, dim=1)

        # Weighted sum: (B, 4, 1) * (B, 4, d_model) -> (B, d_model)
        fused = torch.sum(attn.unsqueeze(-1) * Z, dim=1)

        # Save for diagnostics (detach for safe logging)
        self._last_attention = {
            "id": attn[:, 0].detach(),
            "text": attn[:, 1].detach(),
            "image": attn[:, 2].detach(),
            "time": attn[:, 3].detach(),
        }

        return fused

    def get_last_attention(self) -> Optional[Dict[str, torch.Tensor]]:
        return self._last_attention

In [89]:
# from notbook 06
class ScenarioAwareMMoE(nn.Module):
    """
    Scenario-aware Mixture-of-Experts (MMoE).

    Inputs:
      x:        (B, d_model)   fused representation
      scene_idx:(B,)           scenario indices (0..num_scenes-1)

    Outputs:
      y:        (B, hidden_dim) combined expert output

    Diagnostics:
      get_last_gates(): returns gate weights (B, num_experts)
    """
    def __init__(
        self,
        d_model: int,
        hidden_dim: int,
        num_experts: int,
        num_scenes: int,
        gate_embed_dim: int = 16,
        expert_dropout: float = 0.1
    ):
        super().__init__()

        self.d_model = d_model
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.num_scenes = num_scenes

        # Experts: each expert maps (B, d_model) -> (B, hidden_dim)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, hidden_dim),
                nn.ReLU(),
                nn.Dropout(expert_dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(expert_dropout),
            )
            for _ in range(num_experts)
        ])

        # Scene embedding used for gating
        self.scene_embed = nn.Embedding(num_scenes, gate_embed_dim)

        # Gate network: conditioned on scene embedding (optionally we can include x too later)
        self.gate = nn.Sequential(
            nn.Linear(gate_embed_dim, gate_embed_dim),
            nn.ReLU(),
            nn.Linear(gate_embed_dim, num_experts)
        )

        self._last_gates: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor, scene_idx: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, d_model)
            scene_idx: (B,)
        Returns:
            y: (B, hidden_dim)
        """
        B = x.size(0)

        # Expert outputs: list of (B, hidden_dim) -> stack -> (B, E, hidden_dim)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1)

        # Compute gate weights conditioned on scene_idx
        s_emb = self.scene_embed(scene_idx)           # (B, gate_embed_dim)
        gate_logits = self.gate(s_emb)                # (B, num_experts)
        gates = F.softmax(gate_logits, dim=-1)        # (B, num_experts)

        # Combine experts: (B, E, 1) * (B, E, hidden_dim) -> (B, hidden_dim)
        y = torch.sum(gates.unsqueeze(-1) * expert_outputs, dim=1)

        # Store for diagnostics
        self._last_gates = gates.detach()

        return y

    def get_last_gates(self) -> Optional[torch.Tensor]:
        return self._last_gates

In [90]:
# from notebook 07
class IDEmbeddingLayer(nn.Module):
    """
    Embedding tables for user/item/scene.
    Produces user_emb, item_emb, scene_emb.
    """
    def __init__(self, num_users: int, num_items: int, num_scenes: int,
                 id_dim: int = 256, scene_dim: int = 32, dropout: float = 0.0):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, id_dim)
        self.item_emb = nn.Embedding(num_items, id_dim)
        self.scene_emb = nn.Embedding(num_scenes, scene_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, user_idx: torch.Tensor, item_idx: torch.Tensor, scene_idx: torch.Tensor):
        u = self.dropout(self.user_emb(user_idx))
        i = self.dropout(self.item_emb(item_idx))
        s = self.dropout(self.scene_emb(scene_idx))
        return u, i, s

In [91]:
# from notebook 07
class ItemBlendModule(nn.Module):
    """
    Implements deterministic alpha blending:
      warm -> alpha=1 (use ID item embedding)
      cold -> alpha=0 (use meta embedding)
    """
    def __init__(self):
        super().__init__()
        self._last_alpha = None

    def forward(self, item_id_emb: torch.Tensor, item_meta_emb: torch.Tensor, cold_mask: torch.Tensor):
        alpha = (~cold_mask).float().unsqueeze(1)  # warm=1, cold=0
        item_final = alpha * item_id_emb + (1.0 - alpha) * item_meta_emb

        self._last_alpha = alpha.detach()
        return item_final

    def get_last_alpha(self):
        return self._last_alpha

In [92]:
# from notebook 07
class MetaEmbeddingGenerator(nn.Module):
    """
    Generates meta_item_emb from non-ID modalities: text, image, time
    using projection layers + per-sample modality attention.
    """
    def __init__(
        self,
        text_dim: int,
        image_dim: int,
        time_dim: int,
        meta_dim: int = 256,
        attn_hidden: int = 128,
        dropout: float = 0.1,
        use_scene: bool = False,
        num_scenes: int = 0,
        scene_emb_dim: int = 16
    ):
        super().__init__()
        self.meta_dim = meta_dim
        self.use_scene = use_scene

        # Projections into shared space
        self.proj_text = nn.Sequential(
            nn.Linear(text_dim, meta_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.proj_image = nn.Sequential(
            nn.Linear(image_dim, meta_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.proj_time = nn.Sequential(
            nn.Linear(time_dim, meta_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Optional scene conditioning (lightweight)
        if self.use_scene:
            assert num_scenes > 0, "num_scenes must be provided if use_scene=True"
            self.scene_emb = nn.Embedding(num_scenes, scene_emb_dim)
            attn_in = meta_dim + scene_emb_dim
        else:
            attn_in = meta_dim

        # Attention scoring network (shared across modalities)
        self.attn_mlp = nn.Sequential(
            nn.Linear(attn_in, attn_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(attn_hidden, 1)
        )

        self._last_attention = None  # diagnostics

    def forward(
        self,
        text_vec: torch.Tensor,   # (B, text_dim)
        image_vec: torch.Tensor,  # (B, image_dim)
        time_vec: torch.Tensor,   # (B, time_dim)
        scene_idx: torch.Tensor = None  # optional (B,)
    ) -> torch.Tensor:
        B = text_vec.size(0)

        # Project modalities to (B, meta_dim)
        z_text  = self.proj_text(text_vec)
        z_image = self.proj_image(image_vec)
        z_time  = self.proj_time(time_vec)

        # Stack into (B, M, meta_dim) where M=3 modalities
        Z = torch.stack([z_text, z_image, z_time], dim=1)  # (B, 3, meta_dim)

        if self.use_scene:
            assert scene_idx is not None, "scene_idx required when use_scene=True"
            s = self.scene_emb(scene_idx)                   # (B, scene_emb_dim)
            s = s.unsqueeze(1).expand(B, 3, s.size(-1))     # (B, 3, scene_emb_dim)
            Z_attn_in = torch.cat([Z, s], dim=-1)           # (B, 3, meta_dim+scene_emb_dim)
        else:
            Z_attn_in = Z                                   # (B, 3, meta_dim)

        # Compute attention logits per modality: (B, 3, 1) -> (B, 3)
        attn_logits = self.attn_mlp(Z_attn_in).squeeze(-1)  # (B, 3)
        attn_weights = F.softmax(attn_logits, dim=1)        # (B, 3)

        # Weighted sum to get meta embedding: (B, meta_dim)
        meta_item_emb = torch.sum(Z * attn_weights.unsqueeze(-1), dim=1)

        # Store attention for diagnostics
        self._last_attention = {
            "text": attn_weights[:, 0].detach(),
            "image": attn_weights[:, 1].detach(),
            "time": attn_weights[:, 2].detach()
        }

        return meta_item_emb

    def get_last_attention(self):
        return self._last_attention

In [93]:
class CTRHead(nn.Module):
    def __init__(self, input_dim=256, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)  # logit
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)  # (B,)

In [94]:
class ColdFusionXModel(nn.Module):
    def __init__(self, fusion_config, meta_config, dropout=0.1, scene_dim=32):
        super().__init__()

        # Configs
        self.num_users = int(fusion_config["num_users"])
        self.num_items = int(fusion_config["num_items"])
        self.num_scenes = int(fusion_config["num_scenes"])

        self.text_dim = int(fusion_config["text_dim"])
        self.image_dim = int(fusion_config["image_dim"])
        self.time_dim = int(fusion_config["time_dim"])

        self.d_model = int(fusion_config["d_model"])
        self.hidden_dim = int(fusion_config["hidden_dim"])
        self.num_experts = int(fusion_config["num_experts"])

        self.meta_dim = int(meta_config["meta_dim"])
        assert self.meta_dim == self.d_model, "Recommended: meta_dim should match d_model."

        # 1) ID embeddings (from Notebook 06/07 class)
        self.id_layer = IDEmbeddingLayer(
            num_users=self.num_users,
            num_items=self.num_items,
            num_scenes=self.num_scenes,
            id_dim=self.d_model,
            scene_dim=scene_dim,
            dropout=dropout
        )

        # 2) Meta embedding generator (from Notebook 07 class)
        self.meta_gen = MetaEmbeddingGenerator(
            text_dim=self.text_dim,
            image_dim=self.image_dim,
            time_dim=self.time_dim,
            meta_dim=self.meta_dim,
            attn_hidden=128,
            dropout=dropout,
            use_scene=False
        )

        # 3) α-blend (from Notebook 07 class)
        self.blender = ItemBlendModule()

        # Project concat([user_emb, item_final, scene_emb]) -> d_model
        self.id_proj = nn.Sequential(
            nn.Linear(self.d_model + self.d_model + scene_dim, self.d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # 4) Fusion (from Notebook 06 class)
        self.fusion = ModalityAttentionFusion(
            id_dim=self.d_model,
            text_dim=self.text_dim,
            image_dim=self.image_dim,
            time_dim=self.time_dim,
            d_model=self.d_model,
            attn_hidden=128,
            dropout=dropout
        )

        # 5) Scenario-aware routing (from Notebook 06 class signature)
        self.mmoe = ScenarioAwareMMoE(
            d_model=self.d_model,
            hidden_dim=self.hidden_dim,
            num_experts=self.num_experts,
            num_scenes=self.num_scenes,
            gate_embed_dim=16,
            expert_dropout=dropout
        )

        # 6) CTR head
        self.head = CTRHead(input_dim=self.hidden_dim, hidden_dim=self.hidden_dim, dropout=dropout)

        # Diagnostics handles
        self._last_alpha = None

    def forward(self, user_idx, item_idx, scene_idx, text_vec, image_vec, time_vec, cold_mask):
        # ID embeddings
        user_emb, item_id_emb, scene_emb = self.id_layer(user_idx, item_idx, scene_idx)

        # Meta item embedding
        item_meta_emb = self.meta_gen(text_vec=text_vec, image_vec=image_vec, time_vec=time_vec)

        # Unified item embedding (α blend)
        item_final = self.blender(item_id_emb=item_id_emb, item_meta_emb=item_meta_emb, cold_mask=cold_mask)
        self._last_alpha = self.blender.get_last_alpha()

        # Build id_vec
        id_concat = torch.cat([user_emb, item_final, scene_emb], dim=1)
        id_vec = self.id_proj(id_concat)

        # Fusion + routing
        fused = self.fusion(id_vec=id_vec, text_vec=text_vec, image_vec=image_vec, time_vec=time_vec)
        routed = self.mmoe(fused, scene_idx)

        # Logit
        logit = self.head(routed)
        return logit

    # Diagnostics helpers (match internal storage types)
    def get_last_alpha(self):
        return self._last_alpha

    def get_last_fusion_attention(self):
        return self.fusion.get_last_attention()  # dict with keys id/text/image/time

    def get_last_meta_attention(self):
        return self.meta_gen.get_last_attention()  # dict with keys text/image/time

    def get_last_gates(self):
        return self.mmoe.get_last_gates()  # (B, num_experts)

#### Load checkpoints from Notebook 07 (meta block) + forward pass sanity check
- This ensures the model starts from the saved meta generator weights.

In [95]:
# Instantiate model
model = ColdFusionXModel(fusion_config=fusion_config, meta_config=meta_config, dropout=0.1).to(device)

# Load MetaEmbeddingGenerator checkpoint from Notebook 07
meta_generator_ckpt = os.path.join(meta_artifacts_dir, "meta_generator.pt")
state = torch.load(meta_generator_ckpt, map_location="cpu")
model.meta_gen.load_state_dict(state)
print("Loaded meta_generator weights from:", meta_generator_ckpt)

model.eval()

Loaded meta_generator weights from: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/meta_artifacts/meta_generator.pt


ColdFusionXModel(
  (id_layer): IDEmbeddingLayer(
    (user_emb): Embedding(67615, 256)
    (item_emb): Embedding(59142, 256)
    (scene_emb): Embedding(5, 32)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (meta_gen): MetaEmbeddingGenerator(
    (proj_text): Sequential(
      (0): Linear(in_features=1536, out_features=256, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.1, inplace=False)
    )
    (proj_image): Sequential(
      (0): Linear(in_features=768, out_features=256, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.1, inplace=False)
    )
    (proj_time): Sequential(
      (0): Linear(in_features=5, out_features=256, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.1, inplace=False)
    )
    (attn_mlp): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.1, inplace=False)
      (3): Linear(in_features=128, out_features=1, bias=True)
    )
  )
  (blender): ItemBlendModule()
  (id_proj): Sequential(


In [96]:
batch = next(iter(train_loader))

user_idx  = batch["user_idx"].to(device)
item_idx  = batch["item_idx"].to(device)
scene_idx = batch["scene_idx"].to(device)

text_vec  = batch["text_vec"].to(device)
image_vec = batch["image_vec"].to(device)
time_vec  = batch["time_vec"].to(device)

cold_mask = batch["cold_mask"].to(device)

In [97]:
with torch.no_grad():
    logit = model(
        user_idx=user_idx,
        item_idx=item_idx,
        scene_idx=scene_idx,
        text_vec=text_vec,
        image_vec=image_vec,
        time_vec=time_vec,
        cold_mask=cold_mask
    )

assert torch.isfinite(logit).all(), "Logits contain NaNs/Infs"

In [98]:
# Pull diagnostics
fusion_attn = model.get_last_fusion_attention()  # dict
meta_attn   = model.get_last_meta_attention()    # dict
gates       = model.get_last_gates()             # (B, num_experts)
alpha       = model.get_last_alpha()             # (B,1)

# Check attention sums
fusion_sum = (fusion_attn["id"] + fusion_attn["text"] + fusion_attn["image"] + fusion_attn["time"])
meta_sum   = (meta_attn["text"] + meta_attn["image"] + meta_attn["time"])

print("--- Sanity Forward Pass ---")
print("logit shape:", tuple(logit.shape))
print("alpha shape:", tuple(alpha.shape), "| unique:", torch.unique(alpha).detach().cpu().numpy())

print("fusion attn sum min/max:",
      float(fusion_sum.min().cpu()), float(fusion_sum.max().cpu()))
print("meta attn sum min/max:",
      float(meta_sum.min().cpu()), float(meta_sum.max().cpu()))

print("gates shape:", tuple(gates.shape))
print("gate sum min/max:",
      float(gates.sum(dim=1).min().cpu()), float(gates.sum(dim=1).max().cpu()))

--- Sanity Forward Pass ---
logit shape: (4096,)
alpha shape: (4096, 1) | unique: [0. 1.]
fusion attn sum min/max: 0.9999998807907104 1.0000001192092896
meta attn sum min/max: 0.9999998807907104 1.0000001192092896
gates shape: (4096, 4)
gate sum min/max: 0.9999999403953552 1.0


* Forward pass succeeded and produced logits with the expected shape: logit (1024,).
* α-blend behaves as designed: alpha (1024,1) with unique values [0, 1] confirms deterministic cold/warm blending is active.
* Fusion attention is properly normalised: sums are ~1.0 for every sample (min/max ~1.0).
* Meta attention is properly normalised: sums are ~1.0 for every sample (min/max ~1.0).
* MMoE gating is properly normalised: gates (1024,4) and sums ~1.0.

### Loss, Metrics, and Optimisation

We optimise binary click labels using BCEWithLogitsLoss, which is numerically stable because it operates directly on logits. Model parameters are trained with AdamW (weight decay regularisation), with an optional learning-rate scheduler. During both training and validation, performance is tracked using AUC (primary metric for CTR ranking quality), log loss, and accuracy, including a cold vs warm interaction breakdown to quantify improvements under cold-start conditions.

In [99]:
@torch.no_grad()
def compute_metrics(y_true: np.ndarray, y_logit: np.ndarray):
    """
    y_true: (N,) float/0-1
    y_logit: (N,) raw logits
    """
    y_prob = 1.0 / (1.0 + np.exp(-y_logit))  # sigmoid

    # Manual clipping for numerical safety (replaces eps argument)
    y_prob = np.clip(y_prob, 1e-7, 1.0 - 1e-7)

    auc = None
    if np.unique(y_true).size == 2:
        auc = roc_auc_score(y_true, y_prob)

    ll = log_loss(y_true, y_prob)  # <-- no eps argument
    acc = accuracy_score(y_true, (y_prob >= 0.5).astype(np.int32))

    return {"auc": auc, "logloss": ll, "acc": acc}


@torch.no_grad()
def compute_cold_warm_metrics(y_true: np.ndarray, y_logit: np.ndarray, cold_mask: np.ndarray):
    """
    cold_mask: boolean array True=cold, False=warm
    Returns overall + cold + warm metrics.
    """
    out = {}
    out["overall"] = compute_metrics(y_true, y_logit)

    if cold_mask.any():
        out["cold"] = compute_metrics(y_true[cold_mask], y_logit[cold_mask])
    else:
        out["cold"] = {"auc": None, "logloss": None, "acc": None}

    warm_mask = ~cold_mask
    if warm_mask.any():
        out["warm"] = compute_metrics(y_true[warm_mask], y_logit[warm_mask])
    else:
        out["warm"] = {"auc": None, "logloss": None, "acc": None}

    return out

In [100]:
# Loss (logit-stable)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

# Optimiser
lr = 1e-3
weight_decay = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# additionally, scheduler (safe default: None)
scheduler = None
# Example: StepLR
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.8)

print("Configured:")
print("Loss: BCEWithLogitsLoss")
print(f"Optimiser: AdamW | lr={lr} | weight_decay={weight_decay}")
print("Scheduler:", type(scheduler).__name__ if scheduler is not None else "None")

Configured:
Loss: BCEWithLogitsLoss
Optimiser: AdamW | lr=0.001 | weight_decay=0.0001
Scheduler: None


/tmp/ipython-input-2330916094.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


#### One epoch: train + evaluate (with cold/warm metrics)
* This is a stable “skeleton” loop. It avoids storing everything in GPU memory and collects logits/labels on CPU for metrics.

In [101]:
def run_one_epoch_train(model, loader, optimizer, criterion, device):
    MAX_METRIC_BATCHES = 50  # collect only first ~50 batches for train metrics
    model.train()
    total_loss = 0.0
    n = 0

    # For metrics aggregation
    all_logits = []
    all_labels = []
    all_cold = []

    for batch in tqdm(loader, desc="Train", leave=False):
        user_idx  = batch["user_idx"].to(device, non_blocking=True)
        item_idx  = batch["item_idx"].to(device, non_blocking=True)
        scene_idx = batch["scene_idx"].to(device, non_blocking=True)

        text_vec  = batch["text_vec"].to(device, non_blocking=True)
        image_vec = batch["image_vec"].to(device, non_blocking=True)
        time_vec  = batch["time_vec"].to(device, non_blocking=True)

        cold_mask = batch["cold_mask"].to(device, non_blocking=True)
        y         = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logit = model(
                user_idx=user_idx,
                item_idx=item_idx,
                scene_idx=scene_idx,
                text_vec=text_vec,
                image_vec=image_vec,
                time_vec=time_vec,
                cold_mask=cold_mask
            )
            loss = criterion(logit, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        n += bs

        # Collect on CPU for metrics
        if len(all_logits) < MAX_METRIC_BATCHES:
          all_logits.append(logit.detach().cpu().numpy())
          all_labels.append(y.detach().cpu().numpy())
          all_cold.append(cold_mask.detach().cpu().numpy())

    y_logit = np.concatenate(all_logits, axis=0)
    y_true  = np.concatenate(all_labels, axis=0)
    cold    = np.concatenate(all_cold, axis=0).astype(bool)

    metrics = compute_cold_warm_metrics(y_true=y_true, y_logit=y_logit, cold_mask=cold)
    metrics["train_loss"] = total_loss / max(n, 1)
    return metrics

In [102]:
@torch.no_grad()
def run_one_epoch_eval(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    n = 0

    all_logits = []
    all_labels = []
    all_cold = []

    for batch in tqdm(loader, desc="Val", leave=False):
        user_idx  = batch["user_idx"].to(device, non_blocking=True)
        item_idx  = batch["item_idx"].to(device, non_blocking=True)
        scene_idx = batch["scene_idx"].to(device, non_blocking=True)

        text_vec  = batch["text_vec"].to(device, non_blocking=True)
        image_vec = batch["image_vec"].to(device, non_blocking=True)
        time_vec  = batch["time_vec"].to(device, non_blocking=True)

        cold_mask = batch["cold_mask"].to(device, non_blocking=True)
        y         = batch["label"].to(device, non_blocking=True)

        logit = model(
            user_idx=user_idx,
            item_idx=item_idx,
            scene_idx=scene_idx,
            text_vec=text_vec,
            image_vec=image_vec,
            time_vec=time_vec,
            cold_mask=cold_mask
        )

        loss = criterion(logit, y)

        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        n += bs

        all_logits.append(logit.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())
        all_cold.append(cold_mask.detach().cpu().numpy())

    y_logit = np.concatenate(all_logits, axis=0)
    y_true  = np.concatenate(all_labels, axis=0)
    cold    = np.concatenate(all_cold, axis=0).astype(bool)

    metrics = compute_cold_warm_metrics(y_true=y_true, y_logit=y_logit, cold_mask=cold)
    metrics["val_loss"] = total_loss / max(n, 1)
    return metrics

#### Training loop (few epochs) with logging
* Given the AntM2C dataset size, we will start with 1 epoch for validation and confirm that everything runs, and then scale up.

In [104]:
epochs = 1  # starting small small; increase later once stable

for epoch in range(1, epochs + 1):
    train_metrics = run_one_epoch_train(model, train_loader, optimizer, criterion, device)
    val_metrics   = run_one_epoch_eval(model, val_loader, criterion, device)

    if scheduler is not None:
        scheduler.step()

    def fmt(m):
        # AUC can be None in edge cases
        auc = m["auc"]
        return f"AUC={auc:.4f}" if auc is not None else "AUC=None"

    print(f"\nEpoch {epoch}/{epochs}")
    print(f"Train: loss={train_metrics['train_loss']:.4f} | "
          f"{fmt(train_metrics['overall'])} | logloss={train_metrics['overall']['logloss']:.4f} | acc={train_metrics['overall']['acc']:.4f}")
    print(f"  Cold: {fmt(train_metrics['cold'])} | logloss={train_metrics['cold']['logloss']:.4f} | acc={train_metrics['cold']['acc']:.4f}")
    print(f"  Warm: {fmt(train_metrics['warm'])} | logloss={train_metrics['warm']['logloss']:.4f} | acc={train_metrics['warm']['acc']:.4f}")

    print(f"Val:   loss={val_metrics['val_loss']:.4f} | "
          f"{fmt(val_metrics['overall'])} | logloss={val_metrics['overall']['logloss']:.4f} | acc={val_metrics['overall']['acc']:.4f}")
    print(f"  Cold: {fmt(val_metrics['cold'])} | logloss={val_metrics['cold']['logloss']:.4f} | acc={val_metrics['cold']['acc']:.4f}")
    print(f"  Warm: {fmt(val_metrics['warm'])} | logloss={val_metrics['warm']['logloss']:.4f} | acc={val_metrics['warm']['acc']:.4f}")

Train:   0%|          | 0/729 [00:00<?, ?it/s]/tmp/ipython-input-2384511906.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipython-input-3189717648.py:7: RuntimeWarning: overflow encountered in exp
  y_prob = 1.0 / (1.0 + np.exp(-y_logit))  # sigmoid
/tmp/ipython-input-3189717648.py:7: RuntimeWarning: overflow encountered in exp
  y_prob = 1.0 / (1.0 + np.exp(-y_logit))  # sigmoid



Epoch 1/1
Train: loss=0.3511 | AUC=0.9078 | logloss=0.3572 | acc=0.8392
  Cold: AUC=0.7980 | logloss=0.3294 | acc=0.8684
  Warm: AUC=0.9086 | logloss=0.3582 | acc=0.8381
Val:   loss=0.3450 | AUC=0.9151 | logloss=0.3450 | acc=0.8467
  Cold: AUC=0.8204 | logloss=0.3197 | acc=0.8700
  Warm: AUC=0.9158 | logloss=0.3458 | acc=0.8460


After one full training epoch on A100, ColdFusionX demonstrates strong overall CTR performance (Val AUC ≈ 0.915) with a clear and expected cold–warm gap (cold AUC ≈ 0.82 vs warm AUC ≈ 0.92), confirming that the unified meta-embedding and α-blend design is functioning correctly while highlighting cold-start as a measurable, improvable challenge rather than a failure mode.

In [107]:
!git commit -m "training: complete one full training epoch"